In [ ]:
import torch
from torch import nn
import numpy as np
import math


def generate_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
    return mask.unsqueeze(0).unsqueeze(0).to(device)
def create_padding_mask(seq, pad_token=0):
    mask = (seq != pad_token)   
    return mask.unsqueeze(1).unsqueeze(2)  
    # shape → (batch, 1, 1, seq_len)
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self , d_model , s_len , num_head):
        super(MultiHeadAttentionLayer , self).__init__()
        assert d_model %  num_head == 0  # Check weather the d_model is divisible by number of head 
        self.dv = d_model //  num_head
        self.d_model = d_model
        self.s_len = s_len
        self.num_head = num_head
        self.wq  = nn.Linear(d_model , d_model)
        self.wk = nn.Linear(d_model , d_model)
        self.wv = nn.Linear(d_model , d_model)
        self.wo = nn.Linear(d_model , d_model)
        # Shape - wq , wk ,wv ,wo - (d_model * d_model)
    def resize(self , x:torch.Tensor ):
        batch_size  , seq_len ,  _  = x.size()
        return x.view((batch_size , seq_len , self.num_head , self.dv)).transpose(1,2)
    
    def combine(self , x:torch.Tensor):
        batch_size , _ , seq_len , _ =  x.size()
        return x.transpose(1,2).contiguous().view((batch_size , seq_len , self.d_model)) 
    
    
    def scalar_dot_product(self , q:torch.Tensor,k:torch.Tensor,v:torch.Tensor  , masked:torch.Tensor =  None):
        attention_score:torch.Tensor = torch.matmul(q , k.transpose(2,3)) / math.sqrt(self.dv)
        if masked is not None:
            attention_score = attention_score.masked_fill(~masked ,  float('-inf'))
        softmax = torch.softmax(attention_score , dim=-1)  
        return torch.matmul(softmax , v)

    def forward(self , q:torch.Tensor,k:torch.Tensor,v:torch.Tensor , masked : torch.Tensor = None):
        q = self.resize(self.wq(q))
        k =  self.resize(self.wk(k))
        v = self.resize(self.wv(v))
        attention = self.scalar_dot_product(q,k,v , masked)
     
        return self.wo(self.combine(attention))
        
class AddAndNormalize(nn.Module):
    def __init__(self, eps=1e-6):
        super(AddAndNormalize, self).__init__()
        self.eps = eps

    def forward(self, x, sublayer_output):
        added = x + sublayer_output
        mean = torch.mean(added, dim=-1, keepdim=True)
        var = torch.var(added, dim=-1, keepdim=True)
        return (added - mean) / torch.sqrt(var + self.eps)
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model ):
        super(FeedForwardNetwork, self).__init__()
        self.f1 = nn.Linear( d_model , 4*d_model )
        self.f2 =  nn.Linear(4*d_model ,d_model  )
        self.relu = nn.ReLU()
    def forward(self , x):
        return self.f2(self.relu(self.f1(x)))

class PositionalEncoding:
    def __init__(self, d_model, seq_len):
        position = torch.arange(seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe = torch.zeros(seq_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.encoding = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1)].to(x.device)
class EmbeddingsEncoding(nn.Module):
    def __init__(self  , vocal_size , d_model , seq_size):
        super(EmbeddingsEncoding , self).__init__()
        self.vocal_size =  vocal_size
        self.d_model = d_model
        self.embeddings =  nn.Embedding(vocal_size , d_model)
        self.positionEncoding = PositionalEncoding(d_model , seq_size)
    def forward(self , x ):
        token = self.embeddings(x) * math.sqrt(self.d_model)
        return self.positionEncoding.forward(token)


class EncoderBlock(nn.Module):
    def __init__(self, d_model , seq_len , num_head):
        super(EncoderBlock , self).__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.num_head = num_head
        self.multi_attention_layer = MultiHeadAttentionLayer(d_model , seq_len , num_head)
        self.add_and_normalize = AddAndNormalize()
        self.feed_forward_network = FeedForwardNetwork(d_model )
    def forward(self, x , mask=None):
        attention_output = self.multi_attention_layer(x , x , x , masked = mask)
        add_normalize_output = self.add_and_normalize.forward(x ,  attention_output)
        feed_forward_output = self.feed_forward_network(add_normalize_output)
        add_normalize_output = self.add_and_normalize.forward(add_normalize_output  , feed_forward_output)
        return add_normalize_output
    


class DecoderBlock(nn.Module):
    def __init__(self, d_model , seq_len , num_head):
        super(DecoderBlock , self).__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.num_head = num_head
        self.masked_attention_layer = MultiHeadAttentionLayer(d_model , seq_len , num_head)
        self.add_and_normize_masked = AddAndNormalize()
        self.multi_attention_layer = MultiHeadAttentionLayer(d_model , seq_len , num_head)
        self.add_and_normize_attention = AddAndNormalize()
        self.feed_forward_network =  FeedForwardNetwork(d_model)
        self.add_and_normize_feed = AddAndNormalize()
    def forward(self , x , encoder_value , mask ):
        masked_attention = self.masked_attention_layer(x,x,x , masked=mask)
        add_and_normize_masked_output = self.add_and_normize_masked(x ,masked_attention )
        multi_attention = self.multi_attention_layer( add_and_normize_masked_output , encoder_value , encoder_value)
        add_and_normize_output = self.add_and_normize_attention(add_and_normize_masked_output ,multi_attention )
        feed_forward_output = self.feed_forward_network(add_and_normize_output)
        output = self.add_and_normize_feed(add_and_normize_output ,feed_forward_output )
        return output

class Mask(nn.Module):
    def __init__(self, seq_len):
        super(Mask, self).__init__()
        self.seq_len = seq_len

    def forward(self):
        mask = torch.tril(torch.ones(self.seq_len, self.seq_len)).bool()
        return mask.unsqueeze(0).unsqueeze(0)  # (1,1,seq,seq)

class Transformer(nn.Module):
    def __init__(self , d_model , seq_len , num_head  , vocal_size):
        super(Transformer , self ).__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.num_head = num_head
        self.encoder_layers = nn.ModuleList(
                [EncoderBlock(d_model, seq_len, num_head) for _ in range(4)]
            )

        self.decoder_layers = nn.ModuleList(
            [DecoderBlock(d_model, seq_len, num_head) for _ in range(4)]
        )
        self.linear =  nn.Linear(d_model , vocal_size)
        self.masked = Mask( seq_len)
        self.embedding = EmbeddingsEncoding(vocal_size, d_model, seq_len)
    def forward(self, src, tgt):
        src_mask = create_padding_mask(src).to(src.device)
        tgt_padding_mask = create_padding_mask(tgt).to(src.device)

        src = self.embedding(src)
        tgt = self.embedding(tgt)

        tgt_len = tgt.size(1)

        look_ahead_mask = torch.tril(torch.ones(tgt_len, tgt_len)).bool()
        look_ahead_mask = look_ahead_mask.unsqueeze(0).unsqueeze(0).to(src.device)

        
        combined_mask = look_ahead_mask & tgt_padding_mask
        enc_output = src
        for layer in self.encoder_layers:
            enc_output = layer(enc_output , src_mask )

        dec_output = tgt
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, combined_mask)

        return self.linear(dec_output)



# X = torch.randint(0, vocab_size, (batch, seq_len))

# embedding_layer = EmbeddingsEncoding(vocab_size, d_model, seq_len)

# embedded_X = embedding_layer(X)


Epoch 1, Loss: 2.6661
Epoch 2, Loss: 1.6624
Epoch 3, Loss: 0.9954
Epoch 4, Loss: 0.6035
Epoch 5, Loss: 0.3477
Epoch 6, Loss: 0.2090
Epoch 7, Loss: 0.1287
Epoch 8, Loss: 0.0880
Epoch 9, Loss: 0.0609
Epoch 10, Loss: 0.0395
Epoch 11, Loss: 0.0266
Epoch 12, Loss: 0.0202
Epoch 13, Loss: 0.0171
Epoch 14, Loss: 0.0151
Epoch 15, Loss: 0.0136
Epoch 16, Loss: 0.0121
Epoch 17, Loss: 0.0107
Epoch 18, Loss: 0.0095
Epoch 19, Loss: 0.0084
Epoch 20, Loss: 0.0074
Epoch 21, Loss: 0.0066
Epoch 22, Loss: 0.0059
Epoch 23, Loss: 0.0053
Epoch 24, Loss: 0.0047
Epoch 25, Loss: 0.0043
Epoch 26, Loss: 0.0039
Epoch 27, Loss: 0.0036
Epoch 28, Loss: 0.0033
Epoch 29, Loss: 0.0031
Epoch 30, Loss: 0.0028
Epoch 31, Loss: 0.0027
Epoch 32, Loss: 0.0025
Epoch 33, Loss: 0.0024
Epoch 34, Loss: 0.0022
Epoch 35, Loss: 0.0021
Epoch 36, Loss: 0.0020
Epoch 37, Loss: 0.0019
Epoch 38, Loss: 0.0018
Epoch 39, Loss: 0.0018
Epoch 40, Loss: 0.0017
Epoch 41, Loss: 0.0016
Epoch 42, Loss: 0.0016
Epoch 43, Loss: 0.0015
Epoch 44, Loss: 0.00

In [ ]:
sentences = [
    "i love ai",
    "you love ml",
    "ai is powerful",
    "deep learning is fun"
]
from collections import Counter

def build_vocab(sentences):
    words = " ".join(sentences).split()
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2}
    
    for word in Counter(words):
        vocab[word] = len(vocab)
    
    return vocab

vocab = build_vocab(sentences)
vocab_size = len(vocab)
def tokenize(sentence, vocab):
    tokens = [vocab["<sos>"]]
    tokens += [vocab[word] for word in sentence.split()]
    tokens += [vocab["<eos>"]]
    return tokens
def pad_sequence(seq, max_len, pad_token=0):
    return seq + [pad_token] * (max_len - len(seq))

max_len = 10

data = []
for s in sentences:
    tokens = tokenize(s, vocab)
    tokens = pad_sequence(tokens, max_len)
    data.append(tokens)

data = torch.tensor(data) 


In [ ]:
src = data
tgt = data

tgt_input  = tgt[:, :-1]
tgt_output = tgt[:, 1:]
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = Transformer(d_model=512, seq_len=max_len, num_head=8, vocal_size=vocab_size).to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=0)  
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    epochs = 50

    for epoch in range(epochs):
        model.train()

        src = src.to(device)
        tgt_input = tgt_input.to(device)
        tgt_output = tgt_output.to(device)

        output = model(src, tgt_input)

        output = output.contiguous().view(-1, vocab_size)
        tgt_output = tgt_output.contiguous().view(-1)

        loss = criterion(output, tgt_output)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [7]:
id_to_word = {v: k for k, v in vocab.items()}

In [ ]:
def generate_text(model, input_sentence, vocab, id_to_word, max_len=10, device='cpu'):
    model.eval()

    
    tokens = [vocab["<sos>"]] + [vocab[w] for w in input_sentence.split()]

    src = torch.tensor(tokens).unsqueeze(0).to(device)

    
    tgt = torch.tensor([[vocab["<sos>"]]]).to(device)

    for _ in range(max_len):

        output = model(src, tgt)   

        next_token_logits = output[:, -1, :]  
        next_token = torch.argmax(next_token_logits, dim=-1).item()

        
        tgt = torch.cat([tgt, torch.tensor([[next_token]]).to(device)], dim=1)

        
        if next_token == vocab["<eos>"]:
            break

   
    words = [id_to_word[token] for token in tgt.squeeze().tolist()]

    return " ".join(words)

In [12]:
sentence =  'i love'

output = generate_text(model , sentence , vocab , id_to_word , device=device)

In [13]:
output

'<sos> i love ai <eos>'